# 01 · Data understanding

**Owner:** Charlotte

In [ ]:
import sys
import os

sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import numpy as np

from src.data import load_raw, check_quality, make_split, load_train
from src.data import CONTINUOUS_COLS, BINARY_COLS, SEED

FIG_PATH = os.path.join("..", "results", "figures", "eda")
os.makedirs(FIG_PATH, exist_ok=True)

In [ ]:
df = load_raw()
issues = check_quality(df)

print("\n--- Blood Pressure Issues ---")
print(issues["bp_issues"][["SystolicBP", "DiastolicBP"]].head())

print("\n--- Cholesterol Issues ---")
print(issues["chol_issues"][["CholesterolTotal", "CholesterolLDL", "CholesterolHDL"]].head())

both = issues["bp_issues"].index.intersection(issues["chol_issues"].index)
either = issues["bp_issues"].index.union(issues["chol_issues"].index)
print("\n", len(both), "with both |", len(either), f"with at least one ({len(either)/len(df):.0%})")

**Shape & Completeness**\
The dataset contains no missing values and no duplicate rows, so no imputation or deduplication is required.

**Class Balance**\
The dataset shows a moderate class imbalance: 61.9% of patients are diagnosed with Parkinson's Disease (PD) and 38.1% are not. Since PD is the majority class, this mild imbalance is handled by stratification in the train/test split, ensuring both sets preserve the same ratio.

**Medical Inconsistencies**\
Two types of medically impossible value combinations were identified:
* Blood pressure: 200 observations have a DiastolicBP ≥ SystolicBP, which is physiologically impossible (diastolic pressure must always be lower than systolic pressure).
* Cholesterol: 582 observations have CholesterolLDL + CholesterolHDL > CholesterolTotal, which is impossible as LDL and HDL are components of total cholesterol.
* In total, 734 patients (35%) have at least one inconsistency, of whom 48 are affected by both issues simultaneously.

Although each individual value falls within its documented range:
* Systolic BP: [90–180 mmHg], Diastolic BP: [60–120 mmHg]
* Total cholesterol: [150–300 mg/dL], LDL: [50–200 mg/dL], HDL: [20–100 mg/dL]

their **combinations** are medically unrealistic. As instructed, all observations are retained throughout the pipeline. This is acknowledged as a **known limitation of the synthetic dataset.**

In [ ]:
make_split(df) # creates split.csv — run ONCE!

In [ ]:
X_train, y_train = load_train()
df_train = X_train.copy()
df_train["Diagnosis"] = y_train

In [ ]:
bad_bp = df_train["DiastolicBP"] >= df_train["SystolicBP"]

plt.figure(figsize=(6, 5))
plt.scatter(df_train.loc[~bad_bp, "SystolicBP"], df_train.loc[~bad_bp, "DiastolicBP"], s=8, alpha=0.4, label="Consistent")
plt.scatter(df_train.loc[bad_bp, "SystolicBP"], df_train.loc[bad_bp, "DiastolicBP"], s=8, color="red", label=f"Diastolic ≥ Systolic (n={bad_bp.sum()})")
plt.plot([90, 180], [90, 180], "k--", linewidth=1, label="Diastolic = Systolic")
plt.xlabel("SystolicBP (mmHg)")
plt.ylabel("DiastolicBP (mmHg)")
plt.title("Blood Pressure Inconsistencies (train set)")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(FIG_PATH, "bp_inconsistencies.png"), dpi=150)
plt.show()

**Blood Pressure Inconsistencies (train set) figure**\
The scatter plot confirms the 165 inconsistent observations identified in the quality report (165 of the 200 total fall in the train set). Points on or above the dashed diagonal line (Diastolic = Systolic) are physiologically impossible — all are clustered in the low SystolicBP range (90–120 mmHg) where the documented ranges of Systolic [90–180] 
and Diastolic [60–120] overlap, making impossible combinations more likely by synthetic data construction. All observations are retained as instructed.

In [ ]:
ax = y_train.value_counts().sort_index().plot(kind="bar", color=["steelblue", "salmon"])
ax.set_xticklabels(["Non-PD (0)", "PD (1)"], rotation=0)
ax.set_title("Class Balance (Train Set)")
ax.set_ylabel("Count")
plt.tight_layout()
plt.savefig(os.path.join(FIG_PATH, "class_balance.png"), dpi=150)
plt.show()

**Class Balance (Train Set)** figure above shows: 1,043 PD vs 641 Non-PD; 62/38 split.
Stratified splitting ensures this ratio is preserved across both the train and test sets.

The primary evaluation metric is **F2-score**, which weights recall twice as heavily as precision. This choice reflects an error-cost argument: a missed PD diagnosis (false negative) delays treatment and is clinically far more harmful than an unnecessary referral (false positive). The class imbalance is mild and handled separately by stratification — it is not the reason F2 is chosen.

In [ ]:
fig, axes = plt.subplots(5, 3, figsize=(15, 20))
axes = axes.flatten()

for i, col in enumerate(CONTINUOUS_COLS):
    df_train.boxplot(column=col, by="Diagnosis", ax=axes[i], medianprops={"color": "red"})
    axes[i].set_title(col)
    axes[i].set_xlabel("Diagnosis (0=Non-PD, 1=PD)")

plt.suptitle("Continuous Features by Diagnosis")
plt.tight_layout()
plt.subplots_adjust(top=0.95) 
plt.savefig(os.path.join(FIG_PATH, "continuous_by_diagnosis.png"), dpi=150)
plt.show()

In [ ]:
SYMPTOM_COLS = [
    "Tremor", "Rigidity", "Bradykinesia",
    "PosturalInstability", "SpeechProblems",
    "SleepDisorders", "Constipation"
]

symptom_rates = df_train.groupby("Diagnosis")[SYMPTOM_COLS].mean()
# print(symptom_rates)

symptom_rates.T.plot(kind="bar", figsize=(12, 5), color=["steelblue", "salmon"])
plt.title("Symptom Frequency by Diagnosis")
plt.ylabel("Proportion of patients")
plt.xticks(rotation=45, ha="right")
plt.legend(["Non-PD", "PD"])
plt.tight_layout()
plt.savefig(os.path.join(FIG_PATH, "symptom_frequency.png"), dpi=150)
plt.show()

**Symptom Frequency by Diagnosis**\
The bar chart reveals a clear divide between symptom types:
- The four **cardinal motor symptoms** (Tremor, Rigidity, Bradykinesia and PosturalInstability) show notably higher prevalence in PD patients compared to Non-PD patients. Tremor shows the strongest separation 
  (~54% PD vs ~26% Non-PD).
- The three **non-motor symptoms** (SpeechProblems, SleepDisorders and Constipation) show near-identical prevalence across both groups, suggesting little discriminative value individually.

This distinction motivates splitting the engineered symptom count into two separate features during preprocessing:
- **MotorSymptomCount** = sum of Tremor, Rigidity, Bradykinesia, PosturalInstability (high separation)
- **SymptomCount** = sum of SpeechProblems, SleepDisorders, Constipation (low separation, kept for completeness)

In [ ]:
from scipy.stats import chi2_contingency

OTHER_COLS = ["Gender", "Smoking", "FamilyHistoryParkinsons", "TraumaticBrainInjury", "Hypertension", "Diabetes", "Depression", "Stroke", "Ethnicity", "EducationLevel"]

print("--- Chi-square Test: Other Features vs Diagnosis ---")
results = []
for col in OTHER_COLS:
    p = chi2_contingency(pd.crosstab(df_train[col], df_train["Diagnosis"]))[1]
    rates = df_train.groupby(col)["Diagnosis"].mean().round(2).to_dict()
    results.append({"Feature": col, "PD rate per category": rates, "p-value": round(p, 3)})

print(pd.DataFrame(results).to_string(index=False))

**Other Features (Binary, Categorical, Ordinal)**\
Chi-square tests were used to assess the relationship between each 
binary, categorical and ordinal feature and Diagnosis, as Pearson 
correlation is not appropriate for categorical variables.

Of all features tested, only **Depression** shows a statistically 
significant relationship with Diagnosis (p = 0.002): patients with 
depression have a PD rate of 69% compared to 60% for those without. 
All other features — including Gender, Smoking, FamilyHistoryParkinsons, 
Hypertension, Stroke, Ethnicity and EducationLevel — show no significant 
relationship (all p > 0.05), with PD rates near-identical across 
categories (~61–65%).

This confirms that the binary and categorical features carry little 
individual discriminative signal for Diagnosis in this dataset.


In [ ]:
corr = df_train[CONTINUOUS_COLS + ["Diagnosis"]].corr()

plt.figure(figsize=(14, 10))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Heatmap (Train Set)")
plt.tight_layout()
plt.savefig(os.path.join(FIG_PATH, "correlation_heatmap.png"), dpi=150)
plt.show()

In [ ]:
from sklearn.feature_selection import mutual_info_classif

mi = mutual_info_classif(X_train[CONTINUOUS_COLS], y_train, random_state=SEED)
mi_series = pd.Series(mi, index=CONTINUOUS_COLS).sort_values(ascending=False)
print("--- Mutual Information Scores ---")
print(mi_series.round(3))

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, col in zip(axes, ["UPDRS", "MoCA", "FunctionalAssessment", "Age"]):
    df_train.groupby(pd.qcut(df_train[col], 5))["Diagnosis"].mean().plot(marker="o", ax=ax)
    ax.set_title(f"PD rate by {col} quintile")
    ax.set_ylabel("PD rate")
    ax.set_xlabel(col)
    ax.tick_params(axis="x", rotation=45)

plt.suptitle("Non-linear Patterns — PD Rate by Quintile", y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(FIG_PATH, "nonlinear_patterns.png"), dpi=150, bbox_inches="tight")
plt.show()

**Non-linear Patterns**\
Mutual information (MI) confirms UPDRS (MI = 0.136) as by far the most informative feature, with FunctionalAssessment and MoCA both at MI = 0.045. All other continuous features show near-zero MI scores, including Age (MI = 0.000), suggesting they carry almost no predictive information beyond random chance in this synthetic dataset.

The quintile plots reveal important non-linear threshold patterns:
- **UPDRS**: PD rate jumps sharply from ~20% in the lowest quintile to ~65% in the second quintile, then plateaus at ~75%. Separation occurs mainly at the low end — a low UPDRS score is a strong indicator of Non-PD.
- **MoCA**: PD rate remains relatively stable (~65%) across the lower four quintiles but drops to ~43% in the highest quintile. A high MoCA score (good cognition) is a strong indicator of Non-PD.
- **FunctionalAssessment**: PD rate peaks in the second quintile (~75%) then declines steadily. Higher functional ability is associated with lower PD probability.
- **Age**: The quintile plot shows an almost flat line (~55–65% PD rate across all age groups), consistent with the near-zero MI score (MI = 0.000). Age is included as a literature-backed risk factor but shows no meaningful predictive signal in this dataset.

These non-linear patterns guide where FIS membership functions should be placed and suggest that an RBF kernel may outperform a linear kernel for SVM, given the threshold-like behaviour of UPDRS and MoCA. They also motivate adding a binned (decile) representation of UPDRS, MoCA and FunctionalAssessment in the preprocessing (notebook 02), so the SVM and MLP can learn these thresholds directly.

In [ ]:
RANGES = {
    "Age":                      (50, 90),
    "BMI":                      (15, 40),
    "AlcoholConsumption":       (0, 20),
    "PhysicalActivity":         (0, 10),
    "DietQuality":              (0, 10),
    "SleepQuality":             (4, 10),   
    "SystolicBP":               (90, 180),
    "DiastolicBP":              (60, 120),
    "CholesterolTotal":         (150, 300),
    "CholesterolLDL":           (50, 200),
    "CholesterolHDL":           (20, 100),
    "CholesterolTriglycerides": (50, 400),
    "UPDRS":                    (0, 199),  
    "MoCA":                     (0, 30),
    "FunctionalAssessment":     (0, 10)
}

print("--- Outlier Check (Range-based) ---")
for col, (lo, hi) in RANGES.items():
    n_out = ((df_train[col] < lo) | (df_train[col] > hi)).sum()
    print(f"{col}: {n_out} out of range")

print("\n--- Binary Column Validation ---")
for col in BINARY_COLS:
    unique_vals = df[col].unique()
    valid = set(unique_vals).issubset({0, 1})
    print(f"{col}: {unique_vals} → {'BINARY VALUES' if valid else 'UNEXPECTED VALUES'}")

print("\n--- Ethnicity Validation ---")
print(sorted(df["Ethnicity"].unique().tolist()))

print("\n--- EducationLevel Validation ---")
print(sorted(df["EducationLevel"].unique().tolist()))

**Data Validation & Outlier Inspection**

A range-based check was performed on all 15 continuous features using 
the documented ranges from the attribute description. No values outside 
their documented ranges were found.

Additionally, all binary features (Gender, Smoking, FamilyHistoryParkinsons, 
TraumaticBrainInjury, Hypertension, Diabetes, Depression, Stroke, Tremor, 
Rigidity, Bradykinesia, PosturalInstability, SpeechProblems, SleepDisorders, 
Constipation) contain only valid values of 0 or 1. Ethnicity and 
EducationLevel contain only their expected categories [0, 1, 2, 3].

Note: Z-score and IQR-based outlier methods are not appropriate here — 
all continuous features follow a uniform distribution (by synthetic data 
design), for which the maximum possible Z-score is approximately 1.8. 
A threshold of Z > 3 could therefore never flag any observation. The 
range-based check is the correct approach for uniformly distributed 
synthetic data.

No observations are removed on the basis of outliers.


#
## EDA Findings Summary

#
### Features That Best Separate PD from Non-PD

**Continuous features (correlation + box plot evidence):**
- **UPDRS** (r = +0.40, MI = 0.136): The strongest linear and non-linear predictor of Diagnosis across all features. Box plots confirm clearly higher values in PD patients, consistent with UPDRS being a clinical scale specifically designed to measure Parkinson's severity. Quintile plots show a sharp threshold effect at the low end.
- **FunctionalAssessment** (r = −0.22, MI = 0.045): Clearly lower scores in PD patients, reflecting the progressive motor impairment characteristic of Parkinson's Disease.
- **MoCA** (r = −0.17, MI = 0.045): Clearly lower cognitive scores in PD patients, reflecting the cognitive decline often observed in Parkinson's Disease. Quintile plots show separation mainly at the high end.
- **Age** (r ≈ 0.07, MI = 0.000): Slightly higher in PD patients in box plots, consistent with age being a well-documented risk factor in literature. However, MI score of 0.000 and a flat quintile plot confirm no meaningful predictive signal in this dataset.


**Symptom flags (frequency by diagnosis):**
- **Tremor, Rigidity, Bradykinesia and PosturalInstability** show notably higher prevalence in PD patients. These are the four cardinal motor symptoms of Parkinson's Disease and their elevated frequency in PD patients is clinically expected.

**Other features (chi-square test):**
- **Depression** is the only binary/categorical/ordinal feature with a statistically significant relationship with Diagnosis (p = 0.002): patients with depression have a PD rate of 69%, compared to 60% for patients without depression. All other binary, categorical and ordinal features show no significant relationship (all p > 0.05).

**Note on correlation magnitudes:** All correlations are modest (≤ 0.40), which is expected given the synthetic nature of the dataset. UPDRS has the strongest linear relationship with Diagnosis of all features and is the most informative single predictor.

#
### Anomalies & Limitations
- 200 observations have DiastolicBP ≥ SystolicBP (kept as instructed)
- 582 observations have CholesterolLDL + CholesterolHDL > CholesterolTotal (kept as instructed)
- 734 patients (35%) have at least one inconsistency; 48 have both
- No missing values, no duplicate rows, and all values within documented ranges (range-based check used — Z-score not applicable for uniform distributions, see outlier inspection cell)
- Class imbalance: 61.9% PD vs 38.1% Non-PD — handled by stratification
- F2-score justified by error costs (see class balance cell above)

#
### Suggested FIS Inputs
Based on EDA findings and literature, suggested inputs for the Fuzzy Inference System are:

| Feature | Type | Justification |
|---|---|---|
| UPDRS | Continuous | Strongest predictor (r = +0.40, MI = 0.136), clearly higher in PD; threshold effect at low end |
| MoCA | Continuous | Clearly lower in PD (r = −0.17, MI = 0.045); separation at high end |
| FunctionalAssessment | Continuous | Clearly lower in PD (r = −0.22, MI = 0.045); reflects motor impairment |
| Age | Continuous | Literature-backed risk factor; weak signal in this data (r ≈ 0.07, MI = 0.000) |
| MotorSymptomCount | Continuous (engineered) | Sum of 4 cardinal motor symptoms; r ≈ 0.41, strongest engineered feature |

**Note on excluded candidates:**
- **PosturalInstability** is binary (0/1) and not suitable as a fuzzy input — captured by MotorSymptomCount instead.
- **SleepQuality** shows no meaningful relationship with Diagnosis (MI ≈ 0.000) and is excluded.
- **Depression** shows a statistically significant relationship (p = 0.002) but is binary and therefore not suitable as a direct fuzzy input.